# Explorative Analyse: Arbeitsmarkt Regensburg (Jobcenter) — Teil 13: RLB/ELB/NEF-Struktur

Offener Punkt aus [`ArbeitslosenAnalyse.md`](../MD/ArbeitslosenAnalyse.md): Das Blatt
"1.1 Eckwerte" enthält im Block "Grundsicherung für Arbeitsuchende" neben den bereits seit
Kapitel 1 genutzten Kennzahlen **Bedarfsgemeinschaften (BG)** und **Personen in
Bedarfsgemeinschaften (PERS)** auch die Aufschlüsselung in **Regelleistungsberechtigte
(RLB)**, davon **erwerbsfähige Leistungsberechtigte (ELB)** und **nicht erwerbsfähige
Leistungsberechtigte (NEF)** — bislang nur als Momentaufnahme im Funnel-Chart des
Dashboards sichtbar, nie als eigene Zeitreihe betrachtet.

**Zusammenhang der Kennzahlen:** RLB = ELB + NEF. PERS ist größer als RLB — die Differenz
sind Haushaltsmitglieder, die zur Bedarfsgemeinschaft zählen, aber selbst nicht
leistungsberechtigt sind (z. B. Kinder mit eigenem Einkommen oder Angehörige, deren Bedarf
bereits gedeckt ist).

**Datenquelle:** Wie bei [`12_Unterbeschaeftigung.ipynb`](12_Unterbeschaeftigung.ipynb) liegt
diese Struktur bereits vollständig in [`eckwerte_long.csv`](../Data/processed/eckwerte_long.csv)
vor — kein Neu-Einlesen der Rohdaten nötig. Alle fünf Kennzahlen sind reine SGB-II-Werte ohne
"Insgesamt"-Gegenstück (die Bedarfsgemeinschaft ist ein SGB-II-spezifisches Konzept ohne
Entsprechung im SGB III).

**Hinweis zur Datenaktualität:** In einer früheren Fassung dieses Notebooks wurden Abweichungen
zwischen den Rohdateien April–Juni 2026 und `eckwerte_long.csv` vermutet. Mit der
Datenerweiterung um Juli–September 2026 wurde `eckwerte_long.csv` in
[`01_Exploration.ipynb`](01_Exploration.ipynb) vollständig neu aus allen 21 Rohdateien erzeugt;
dabei hat sich kein einziger historischer Wert verändert, und ein gezielter Abgleich der
RLB/ELB/NEF-Werte für April–Juni 2026 zeigt exakte Übereinstimmung. Rohdaten und aufbereitete
Werte sind also konsistent.

In [1]:
from pathlib import Path

import pandas as pd
import plotly.graph_objects as go

PROCESSED_DIR = Path("..") / "Data" / "processed"
ABBILDUNGEN_DIR = Path("..") / "Docs" / "Abbildungen"

df_long = pd.read_csv(PROCESSED_DIR / "eckwerte_long.csv")
df_long["period"] = pd.PeriodIndex(df_long["period"], freq="M")

grundsicherung = df_long[df_long["kategorie"] == "Grundsicherung für Arbeitsuchende"].copy()

wide = grundsicherung.pivot_table(index="period", columns="merkmal", values="sgb2").sort_index()
wide = wide.rename(columns={
    "Bedarfsgemeinschaften (BG)": "bg",
    "Personen in Bedarfsgemeinschaften (PERS)": "pers",
    "dar. Regelleistungsberechtigte (RLB)": "rlb",
    "dav. erwerbsfähige Leistungsberechtigte (ELB)": "elb",
    "nicht erwerbsfähige Leistungsberechtigte (NEF)": "nef",
})
wide.head(3)

merkmal,bg,pers,rlb,elb,nef
period,,,,,
2025-01,2797.303381,5488.033274,5202.270477,3730.590852,1471.679625
2025-02,2755.768665,5434.992911,5147.992911,3710.211568,1437.781343
2025-03,2781.436728,5474.933615,5178.933615,3746.433943,1432.499672


## 1. Zeitreihe: RLB, ELB und NEF im Vergleich

Wie entwickeln sich die drei Kennzahlen hinter dem Regelleistungsberechtigten-Begriff über
die Zeit?

In [2]:
fig = go.Figure()
for spalte, name, farbe in [
    ("rlb", "Regelleistungsberechtigte (RLB)", "#1e3a8a"),
    ("elb", "davon erwerbsfähig (ELB)", "#2563eb"),
    ("nef", "nicht erwerbsfähig (NEF)", "#dc2626"),
]:
    fig.add_trace(go.Scatter(
        x=wide.index.to_timestamp(), y=wide[spalte], name=name,
        mode="lines+markers", line=dict(width=2.5), marker=dict(size=5),
    ))
fig.update_layout(
    title="Regelleistungsberechtigte (RLB): erwerbsfähig (ELB) vs. nicht erwerbsfähig (NEF)",
    xaxis_title="Berichtsmonat", yaxis_title="Personen", template="plotly_white",
    height=450, hovermode="x unified", legend=dict(orientation="h", y=1.15),
)
fig.write_image(ABBILDUNGEN_DIR / "abb38_rlb_elb_nef_zeitreihe.png", scale=2)
fig

**Befund:** Alle drei Kennzahlen sind über den Beobachtungszeitraum rückläufig — RLB von 5.202 auf 4.591 (−11,8 %), ELB von 3.731 auf 3.355 (−10,1 %) und NEF von 1.472 auf 1.236 (−16,0 %). Allein im September 2026 sinken alle drei Werte deutlich (RLB −4,1 % gegenüber August). Die nicht erwerbsfähigen Leistungsberechtigten (im Wesentlichen Kinder) gehen damit relativ am stärksten zurück — stärker sogar als die erwerbsfähigen ELB, auf deren Basis auch die Aufstockerquote aus Kapitel 7 berechnet wird.

## 2. NEF-Anteil an den Regelleistungsberechtigten

Welchen Anteil an allen Regelleistungsberechtigten haben die nicht erwerbsfähigen
Leistungsberechtigten (im Wesentlichen Kinder unter 15 Jahren in der Bedarfsgemeinschaft)?

In [3]:
nef_anteil = (wide["nef"] / wide["rlb"] * 100)

fig = go.Figure()
fig.add_trace(go.Scatter(
    x=wide.index.to_timestamp(), y=nef_anteil, mode="lines+markers",
    line=dict(color="#7c3aed", width=2.5), name="NEF-Anteil an RLB",
))
fig.update_layout(
    title="Anteil nicht erwerbsfähiger Leistungsberechtigter (NEF) an allen RLB",
    xaxis_title="Berichtsmonat", yaxis_title="Anteil (%)", yaxis_ticksuffix="%",
    template="plotly_white", height=420,
)
fig

**Befund:** Der NEF-Anteil an allen Regelleistungsberechtigten sinkt leicht von 28,3 % auf 26,9 % (Spanne 26,9–28,3 %, Tiefstwert im September 2026) — passend zum stärkeren relativen Rückgang der NEF-Zahl gegenüber der ELB-Zahl.

## 3. Nicht leistungsberechtigte Haushaltsmitglieder: PERS minus RLB

Die Differenz zwischen allen Personen in Bedarfsgemeinschaften (PERS) und den
Regelleistungsberechtigten (RLB) zeigt, wie viele Haushaltsmitglieder zur Bedarfsgemeinschaft
zählen, ohne selbst leistungsberechtigt zu sein.

In [4]:
differenz = wide["pers"] - wide["rlb"]
differenz_anteil = (differenz / wide["pers"] * 100)

fig = go.Figure()
fig.add_trace(go.Bar(
    x=wide.index.to_timestamp(), y=differenz, name="PERS − RLB",
    marker_color="#f59e0b", opacity=0.75,
))
fig.add_trace(go.Scatter(
    x=wide.index.to_timestamp(), y=differenz_anteil, name="Anteil an PERS (%)",
    mode="lines+markers", line=dict(color="#b45309", width=2.5), yaxis="y2",
))
fig.update_layout(
    title="Nicht leistungsberechtigte Haushaltsmitglieder (PERS − RLB)",
    xaxis_title="Berichtsmonat", yaxis=dict(title="Personen"),
    yaxis2=dict(title="Anteil an PERS (%)", overlaying="y", side="right", ticksuffix="%"),
    template="plotly_white", height=450, legend=dict(orientation="h", y=1.15),
)
fig.write_image(ABBILDUNGEN_DIR / "abb39_rlb_elb_nef_differenz.png", scale=2)
fig

**Befund:** Während Bedarfsgemeinschaften, Personen in BG und Regelleistungsberechtigte insgesamt rückläufig sind, steigt die Zahl der nicht leistungsberechtigten Haushaltsmitglieder gegen den Trend: von 286 auf 295 Personen (Höchststand 316 im Mai 2026), ihr Anteil an allen Personen in Bedarfsgemeinschaften von 5,2 % auf 6,0 %. Absolut ist die Zahl seit Mai 2026 wieder etwas zurückgegangen, anteilig bleibt sie wegen der insgesamt sinkenden Personenzahl auf dem höheren Niveau. Ein gegenläufiger Trend, der in den bisher genutzten Kennzahlen BG und PERS allein nicht sichtbar war.

## 4. Struktur der Bedarfsgemeinschaften im letzten Berichtsmonat

Wie setzt sich die Zahl der Personen in Bedarfsgemeinschaften im letzten Berichtsmonat
zusammen?

In [5]:
letzter_monat = wide.index.max()
elb_letzter = wide.loc[letzter_monat, "elb"]
nef_letzter = wide.loc[letzter_monat, "nef"]
nicht_lb_letzter = wide.loc[letzter_monat, "pers"] - wide.loc[letzter_monat, "rlb"]

fig = go.Figure(go.Pie(
    labels=["erwerbsfähig (ELB)", "nicht erwerbsfähig (NEF)", "nicht leistungsberechtigt"],
    values=[elb_letzter, nef_letzter, nicht_lb_letzter],
    hole=0.5, marker=dict(colors=["#2563eb", "#dc2626", "#cbd5e1"]),
))
fig.update_layout(
    title=f"Struktur der Personen in Bedarfsgemeinschaften ({letzter_monat})",
    template="plotly_white", height=450,
)
fig.write_image(ABBILDUNGEN_DIR / "abb40_rlb_elb_nef_struktur.png", scale=2)
fig

**Befund:** Im letzten Berichtsmonat entfallen von den Personen in Bedarfsgemeinschaften rund 69 % auf erwerbsfähige Leistungsberechtigte (ELB), 25 % auf nicht erwerbsfähige Leistungsberechtigte (NEF, überwiegend Kinder) und die restlichen rund 6 % auf nicht leistungsberechtigte Haushaltsmitglieder.

## 5. Zusammenfassung

Die Aufschlüsselung von Bedarfsgemeinschaften und Personen in BG in Regelleistungsberechtigte (RLB), davon erwerbsfähige (ELB) und nicht erwerbsfähige (NEF) Leistungsberechtigte zeigt ein differenzierteres Bild als die bisher genutzten Summenkennzahlen: Alle drei Kennzahlen sind rückläufig, NEF dabei mit −16,0 % am stärksten. Der auffälligste Befund ist jedoch gegenläufig — die Zahl der nicht leistungsberechtigten Haushaltsmitglieder (PERS − RLB) steigt sowohl absolut (286 → 295 Personen, Höchststand 316 im Mai 2026) als auch anteilig (5,2 % → 6,0 %), obwohl alle anderen Bedarfsgemeinschafts-Kennzahlen sinken. Diese Struktur war bislang nur als Momentaufnahme im Funnel-Chart des Dashboards sichtbar und wird hier erstmals als eigene Zeitreihe ausgewertet.

In [6]:
export = wide.copy()
export.index.name = "period"
export.to_csv(PROCESSED_DIR / "rlb_elb_nef_sgb2.csv")
print("Gespeichert:", PROCESSED_DIR / "rlb_elb_nef_sgb2.csv")

Gespeichert: ..\Data\processed\rlb_elb_nef_sgb2.csv
